In [137]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50.csv"
)

df = pd.read_csv(data_file)

print("Dataset loaded!")
print("Shape:", df.shape)
print("Patients:", df["caseid"].nunique())

Dataset loaded!
Shape: (12515, 23)
Patients: 50


In [138]:
patient_ids = np.array(
    sorted(df["caseid"].unique())
)

print("Number of patients:", len(patient_ids))
print(patient_ids)

Number of patients: 50
[ 1  3  4  7 10 12 13 14 16 17 19 20 22 24 25 26 27 28 29 31 32 34 38 41
 43 44 46 49 50 51 52 53 55 56 58 59 60 61 64 65 66 67 68 69 70 74 75 76
 77 79]


In [139]:
train_ids, temp_ids = train_test_split(
    patient_ids,
    test_size=0.30,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

In [140]:
print("Training patients:", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients:", len(test_ids))

Training patients: 35
Validation patients: 7
Test patients: 8


In [141]:
train_val_overlap = (
    set(train_ids) & set(val_ids)
)

train_test_overlap = (
    set(train_ids) & set(test_ids)
)

val_test_overlap = (
    set(val_ids) & set(test_ids)
)

print("Train/Validation overlap:", train_val_overlap)
print("Train/Test overlap:", train_test_overlap)
print("Validation/Test overlap:", val_test_overlap)

Train/Validation overlap: set()
Train/Test overlap: set()
Validation/Test overlap: set()


In [142]:
train_df = df[
    df["caseid"].isin(train_ids)
].copy()

val_df = df[
    df["caseid"].isin(val_ids)
].copy()

test_df = df[
    df["caseid"].isin(test_ids)
].copy()

In [143]:
print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print()

print(
    "Training patients:",
    train_df["caseid"].nunique()
)

print(
    "Validation patients:",
    val_df["caseid"].nunique()
)

print(
    "Test patients:",
    test_df["caseid"].nunique()
)

Training rows: 8789
Validation rows: 1860
Test rows: 1866

Training patients: 35
Validation patients: 7
Test patients: 8


In [144]:
df["split"] = "unassigned"

df.loc[
    df["caseid"].isin(train_ids),
    "split"
] = "train"

df.loc[
    df["caseid"].isin(val_ids),
    "split"
] = "validation"

df.loc[
    df["caseid"].isin(test_ids),
    "split"
] = "test"

In [145]:
df.groupby("split")["caseid"].nunique()

split
test           8
train         35
validation     7
Name: caseid, dtype: int64

In [146]:
print(
    "Unassigned rows:",
    (df["split"] == "unassigned").sum()
)

Unassigned rows: 0


In [147]:
split_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50_split.csv"
)

df.to_csv(
    split_file,
    index=False
)

print("Split dataset saved to:")
print(split_file)

Split dataset saved to:
c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\data\processed\multi_patient_50_split.csv


In [148]:
baseline_val = val_df.dropna(
    subset=[
        "spo2",
        "future_spo2_5min"
    ]
).copy()

print(
    "Validation rows usable for baseline:",
    len(baseline_val)
)

Validation rows usable for baseline: 1748


In [149]:
y_val_actual = (
    baseline_val["future_spo2_5min"]
)

y_val_baseline = (
    baseline_val["spo2"]
)

In [150]:
baseline_mae = mean_absolute_error(
    y_val_actual,
    y_val_baseline
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_val_actual,
        y_val_baseline
    )
)

baseline_r2 = r2_score(
    y_val_actual,
    y_val_baseline
)

print("=== PERSISTENCE BASELINE ===")
print()

print(
    f"MAE:  {baseline_mae:.3f}"
)

print(
    f"RMSE: {baseline_rmse:.3f}"
)

print(
    f"R²:   {baseline_r2:.3f}"
)

=== PERSISTENCE BASELINE ===

MAE:  0.267
RMSE: 1.657
R²:   -0.303


In [151]:
baseline_results = pd.DataFrame({
    "model": [
        "Persistence Baseline"
    ],
    "MAE": [
        baseline_mae
    ],
    "RMSE": [
        baseline_rmse
    ],
    "R2": [
        baseline_r2
    ]
})

baseline_results

,model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086


In [152]:
baseline_file = (
    project_root
    / "data"
    / "processed"
    / "baseline_results.csv"
)

baseline_results.to_csv(
    baseline_file,
    index=False
)

print("Baseline results saved!")

Baseline results saved!


In [153]:
baseline_results

,model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
